# Legacy Access — Service Principal Mount



In [0]:
SECRET_SCOPE = "default2"

tenant_id = dbutils.secrets.get(scope=SECRET_SCOPE, key="tenant-id")
client_id = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-databricks-adls-appid")
client_secret = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-databricks-adls-appkey")

print("Client ID loaded:", bool(client_id))
print("Client secret loaded:", bool(client_secret))
print("Tenant ID loaded:", bool(tenant_id))

In [0]:
storage_account = "dlspl21databricks"
container = "ivanrazumovskyi"
source = (
    f"abfss://{container}@"
    f"{storage_account}.dfs.core.windows.net/"
)
mount_point = f"/mnt/{container}"

configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

existing_mounts = {m.mountPoint for m in dbutils.fs.mounts()}

if mount_point not in existing_mounts:
    dbutils.fs.mount(
        source=source,
        mount_point=mount_point,
        extra_configs=configs
    )
    print(f"Mounted {source} at {mount_point}")
else:
    print(f"{mount_point} is already mounted")

## Result: mount blocked by cluster policy

Mounting failed - DBFS root and legacy mounts are disabled on this workspace.